# MiroFish MS POC - Validation Notebook
Test each step of the pipeline in your Dataiku environment.

**Steps:**
1. Check dependencies (networkx, pandas)
2. Test LLM connection (Dataiku LLM Mesh)
3. Text extraction & chunking
4. Entity/relationship extraction via LLM
5. Build NetworkX graph
6. Query the graph
7. Generate a mini report

## Step 1: Check Dependencies

In [ ]:
import json

# Check required packages
deps = {}
for pkg in ["networkx", "pandas", "dataiku"]:
    try:
        __import__(pkg)
        deps[pkg] = "OK"
    except ImportError:
        deps[pkg] = "MISSING"

# Optional
for pkg in ["fitz"]:
    try:
        __import__(pkg)
        deps[f"{pkg} (PyMuPDF)"] = "OK"
    except ImportError:
        deps[f"{pkg} (PyMuPDF)"] = "MISSING (optional - PDF support)"

for k, v in deps.items():
    print(f"  {k}: {v}")

## Step 2: Test LLM Connection

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_default_project()
llm = project.get_llm("openaiLMSOpenAI:gpt-4o")

# Quick test
completion = llm.new_completion()
completion.with_message("Say 'LLM connection OK' and nothing else.")
resp = completion.execute()
print(resp.text)
print("\n--- LLM connection verified ---")

## Step 3: Text Extraction & Chunking
Using sample text. Replace with file upload later.

In [ ]:
SAMPLE_TEXT = """
Morgan Stanley's wealth management division reported strong Q4 2025 results, 
driven by increased client assets and higher fee-based revenues. CEO Ted Pick 
highlighted the firm's strategic focus on integrating technology across its 
advisory platform. The division saw net new assets of $120 billion for the year.

Meanwhile, the institutional securities group faced headwinds from lower trading 
volumes in fixed income markets. CFO Sharon Yeshaya noted that the firm is 
investing heavily in AI-driven risk analytics to improve trading desk performance.

Competitor Goldman Sachs also reported mixed results, with its asset management 
division outperforming while investment banking revenues declined. JPMorgan Chase 
continued to lead in overall revenue, benefiting from its diversified business model.

Industry analysts from Barclays and UBS forecast that wealth management will 
remain the key growth driver for large banks in 2026, as interest rate cuts 
are expected to boost asset valuations and client activity.
"""


def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            for sep in [". ", "\n\n", "\n", " "]:
                last_sep = text[start:end].rfind(sep)
                if last_sep > chunk_size * 0.5:
                    end = start + last_sep + len(sep)
                    break
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks


chunks = chunk_text(SAMPLE_TEXT.strip())
print(f"Total text length: {len(SAMPLE_TEXT.strip())} chars")
print(f"Number of chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ({len(c)} chars) ---")
    print(c[:200] + "..." if len(c) > 200 else c)

## Step 4: Entity & Relationship Extraction via LLM

In [ ]:
import re

EXTRACTION_PROMPT = """Analyze the following text and extract all entities and relationships.

Return a JSON object with exactly this structure:
{{
  "entities": [
    {{"name": "Entity Name", "type": "Person|Organization|Division|Metric|Event", "description": "brief description"}}
  ],
  "relationships": [
    {{"source": "Entity Name", "target": "Entity Name", "relation": "RELATIONSHIP_TYPE", "description": "brief description"}}
  ]
}}

Entity types: Person, Organization, Division, Metric, Event, Technology, Market
Relationship types: WORKS_AT, LEADS, PART_OF, COMPETES_WITH, REPORTS, INVESTS_IN, FORECASTS, IMPACTS

Be thorough. Extract every entity and relationship mentioned.
Return ONLY valid JSON, no markdown fences, no extra text.

Text:
{text}
"""

# Step A: First, let's see exactly what the LLM returns
completion = llm.new_completion()
completion.with_message(EXTRACTION_PROMPT.format(text=SAMPLE_TEXT.strip()))
resp = completion.execute()

raw = resp.text
print("=== RAW repr (first 500 chars) ===")
print(repr(raw[:500]))
print()
print("=== RAW print (first 500 chars) ===")
print(raw[:500])

In [ ]:
# DEBUG: inspect LLM response format
completion = llm.new_completion()
completion.with_message('Return exactly: {"test": 123}')
resp = completion.execute()

print("type(resp):", type(resp))
print("dir(resp):", dir(resp))
print("---")
print("type(resp.text):", type(resp.text))
print("repr(resp.text):", repr(resp.text[:300]))

## Step 5: Build NetworkX Graph

In [ ]:
import networkx as nx


def build_graph(extraction_result):
    """Build a NetworkX graph from extracted entities and relationships."""
    G = nx.DiGraph()
    
    # Add nodes
    for entity in extraction_result["entities"]:
        G.add_node(
            entity["name"],
            type=entity["type"],
            description=entity["description"]
        )
    
    # Add edges
    for rel in extraction_result["relationships"]:
        # Ensure source and target nodes exist
        if rel["source"] not in G:
            G.add_node(rel["source"], type="Unknown", description="")
        if rel["target"] not in G:
            G.add_node(rel["target"], type="Unknown", description="")
        
        G.add_edge(
            rel["source"],
            rel["target"],
            relation=rel["relation"],
            description=rel["description"]
        )
    
    return G


G = build_graph(result)
print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nNodes:")
for node, data in G.nodes(data=True):
    print(f"  [{data.get('type', '?')}] {node}")
print(f"\nEdges:")
for src, tgt, data in G.edges(data=True):
    print(f"  {src} --[{data['relation']}]--> {tgt}")

## Step 6: Query the Graph
Simple graph queries to retrieve context for report generation.

In [ ]:
def get_node_context(G, node_name):
    """Get all information about a node and its neighbors."""
    if node_name not in G:
        return None
    
    info = {"name": node_name, **G.nodes[node_name]}
    
    # Outgoing relationships
    info["outgoing"] = [
        {"target": tgt, **data}
        for _, tgt, data in G.out_edges(node_name, data=True)
    ]
    
    # Incoming relationships
    info["incoming"] = [
        {"source": src, **data}
        for src, _, data in G.in_edges(node_name, data=True)
    ]
    
    return info


def get_full_context(G):
    """Get a text summary of the entire graph for LLM context."""
    lines = ["Knowledge Graph Summary:", ""]
    
    # Group nodes by type
    by_type = {}
    for node, data in G.nodes(data=True):
        t = data.get("type", "Unknown")
        by_type.setdefault(t, []).append((node, data))
    
    lines.append("ENTITIES:")
    for entity_type, nodes in by_type.items():
        lines.append(f"  {entity_type}:")
        for name, data in nodes:
            lines.append(f"    - {name}: {data.get('description', '')}")
    
    lines.append("\nRELATIONSHIPS:")
    for src, tgt, data in G.edges(data=True):
        lines.append(f"  {src} --[{data['relation']}]--> {tgt}: {data.get('description', '')}")
    
    return "\n".join(lines)


# Test: query a specific entity
print("=== Query: Morgan Stanley ===")
ctx = get_node_context(G, "Morgan Stanley")
if ctx:
    print(json.dumps(ctx, indent=2))
else:
    # Try to find a close match
    for n in G.nodes():
        if "morgan" in n.lower():
            ctx = get_node_context(G, n)
            print(f"Found: {n}")
            print(json.dumps(ctx, indent=2))
            break

print("\n=== Full Graph Context ===")
full_ctx = get_full_context(G)
print(full_ctx)

## Step 7: Generate a Mini Report

In [ ]:
REPORT_PROMPT = """You are an analyst generating a brief research report.

Based on the following knowledge graph data, write a short analysis report.

{graph_context}

Original source text:
{source_text}

Write a report with these sections:
1. Executive Summary (2-3 sentences)
2. Key Entities & Their Roles
3. Key Relationships & Dynamics
4. Outlook & Implications

Keep it concise and professional.
"""

completion = llm.new_completion()
completion.with_message(
    REPORT_PROMPT.format(
        graph_context=full_ctx,
        source_text=SAMPLE_TEXT.strip()
    )
)
report_resp = completion.execute()

print("=== GENERATED REPORT ===")
print(report_resp.text)

## Step 8: Interactive Q&A (Bonus)
Ask questions about the graph data.

In [ ]:
def ask_question(llm, graph, source_text, question):
    """Ask a question and get an answer grounded in graph + source data."""
    ctx = get_full_context(graph)
    
    prompt = f"""You are a research assistant. Answer the question using ONLY the provided data.
If the answer is not in the data, say so.

Knowledge Graph:
{ctx}

Source Text:
{source_text}

Question: {question}

Answer concisely."""
    
    completion = llm.new_completion()
    completion.with_message(prompt)
    resp = completion.execute()
    return resp.text


# Test questions
questions = [
    "What is Morgan Stanley investing in?",
    "Who are Morgan Stanley's competitors mentioned here?",
    "What is the industry outlook for 2026?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_question(llm, G, SAMPLE_TEXT.strip(), q)}")
    print()

## Results Summary
If all cells above ran successfully, your environment supports:
- Dataiku LLM Mesh connection
- NetworkX graph building
- LLM-based entity extraction
- Graph querying
- Report generation
- Interactive Q&A

**Next step:** Build this into a Dataiku webapp.